# Preprocess Datasets

preprocess english + italian data

In [1]:
from pathlib import Path
import pandas as pd

from utils.preprocessing import SentenceSplitPreprocessor

In [2]:
RAW_ROOT = Path("./data/raw")
PROCESSED_ROOT = Path("./data/processed")

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

prep = SentenceSplitPreprocessor(window=50)

In [3]:
raw_files = sorted(RAW_ROOT.glob("*/*.sent_split"))

print(f"total raw files found: {len(raw_files)}")
for path in raw_files:
    print(path)

total raw files found: 22
data\raw\UD_English-EWT\en_ewt-ud-dev.sent_split
data\raw\UD_English-EWT\en_ewt-ud-test.sent_split
data\raw\UD_English-EWT\en_ewt-ud-train.sent_split
data\raw\UD_English-GUM\en_gum-ud-dev.sent_split
data\raw\UD_English-GUM\en_gum-ud-test.sent_split
data\raw\UD_English-GUM\en_gum-ud-train.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-dev.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-test.sent_split
data\raw\UD_English-ParTUT\en_partut-ud-train.sent_split
data\raw\UD_English-PUD\en_pud-ud-test.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-dev.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-test.sent_split
data\raw\UD_Italian-ISDT\it_isdt-ud-train.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-dev.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-test.sent_split
data\raw\UD_Italian-MarkIT\it_markit-ud-train.sent_split
data\raw\UD_Italian-ParTUT\it_partut-ud-dev.sent_split
data\raw\UD_Italian-ParTUT\it_partut-ud-test.sent_split
data\raw\UD_Italian-P

In [4]:
#sanity check
sample_path = raw_files[0]

result = prep.process_file(sample_path)
df_sample = result["df"]

print("sample file:", sample_path)
print("shape:", df_sample.shape)
print(df_sample.head())
print(df_sample["label"].value_counts())

sample file: data\raw\UD_English-EWT\en_ewt-ud-dev.sent_split
shape: (5035, 9)
          doc_id  char_idx punct  \
0  en_ewt-ud-dev        31    \n   
1  en_ewt-ud-dev        32    \n   
2  en_ewt-ud-dev       112    \n   
3  en_ewt-ud-dev       153     .   
4  en_ewt-ud-dev       180     .   

                                        left_context  \
0                    From the AP comes this story :    
1                  From the AP comes this story : \n   
2  inated two individuals to replace retiring jur...   
3  g jurists\non federal courts in the Washington...   
4   in the Washington area. Bush nominated Jennif...   

                                       right_context  \
0  \nPresident Bush on Tuesday nominated two indi...   
1  President Bush on Tuesday nominated two indivi...   
2  on federal courts in the Washington area. Bush...   
3   Bush nominated Jennifer M. Anderson\nfor a 15...   
4   Anderson\nfor a 15-year term as associate jud...   

                              

In [5]:
def make_output_path(raw_path: Path, processed_root: Path) -> Path:
    dataset_name = raw_path.parent.name
    out_dir = processed_root / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = raw_path.stem + ".csv"
    return out_dir / out_name

In [7]:
summary_rows = []

for raw_path in raw_files:
    result = prep.process_file(raw_path)
    df = result["df"]

    out_path = make_output_path(raw_path, PROCESSED_ROOT)
    df.to_csv(out_path, index=False, encoding="utf-8")

    summary_rows.append({
        "dataset": raw_path.parent.name,
        "raw_file": raw_path.name,
        "processed_file": out_path.name,
        "n_rows": len(df),
        "n_positive": int(df["label"].sum()),
        "n_negative": int((df["label"] == 0).sum()),
        "clean_text_len": len(result["clean_text"]),
        "newline_only_boundaries": len(result["non_punct_boundary_indices"]),
    })

    print(f"saved: {out_path} | rows={len(df)}")

saved: data\processed\UD_English-EWT\en_ewt-ud-dev.csv | rows=5035
saved: data\processed\UD_English-EWT\en_ewt-ud-test.csv | rows=4994
saved: data\processed\UD_English-EWT\en_ewt-ud-train.csv | rows=30337
saved: data\processed\UD_English-GUM\en_gum-ud-dev.csv | rows=4217
saved: data\processed\UD_English-GUM\en_gum-ud-test.csv | rows=4126
saved: data\processed\UD_English-GUM\en_gum-ud-train.csv | rows=27578
saved: data\processed\UD_English-ParTUT\en_partut-ud-dev.csv | rows=336
saved: data\processed\UD_English-ParTUT\en_partut-ud-test.csv | rows=390
saved: data\processed\UD_English-ParTUT\en_partut-ud-train.csv | rows=4748
saved: data\processed\UD_English-PUD\en_pud-ud-test.csv | rows=3064
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-dev.csv | rows=1340
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-test.csv | rows=1181
saved: data\processed\UD_Italian-ISDT\it_isdt-ud-train.csv | rows=31386
saved: data\processed\UD_Italian-MarkIT\it_markit-ud-dev.csv | rows=1015
saved: data\proces

In [8]:
summary_df = pd.DataFrame(summary_rows).sort_values(["dataset", "raw_file"]).reset_index(drop=True)
summary_df

,dataset,raw_file,processed_file,n_rows,n_positive,n_negative,clean_text_len,newline_only_boundaries
0,UD_English-EWT,en_ewt-ud-dev.sent_split,en_ewt-ud-dev.csv,5035,1915,3120,126839,507
1,UD_English-EWT,en_ewt-ud-test.sent_split,en_ewt-ud-test.csv,4994,1963,3031,126372,634
2,UD_English-EWT,en_ewt-ud-train.sent_split,en_ewt-ud-train.csv,30337,12004,18333,1012902,2380
3,UD_English-GUM,en_gum-ud-dev.sent_split,en_gum-ud-dev.csv,4217,1566,2651,137961,159
4,UD_English-GUM,en_gum-ud-test.sent_split,en_gum-ud-test.csv,4126,1454,2672,144356,170
5,UD_English-GUM,en_gum-ud-train.sent_split,en_gum-ud-train.csv,27578,10204,17374,871514,1134
6,UD_English-PUD,en_pud-ud-test.sent_split,en_pud-ud-test.csv,3064,1000,2064,111913,0
7,UD_English-ParTUT,en_partut-ud-dev.sent_split,en_partut-ud-dev.csv,336,156,180,14186,9
8,UD_English-ParTUT,en_partut-ud-test.sent_split,en_partut-ud-test.csv,390,153,237,18504,2
9,UD_English-ParTUT,en_partut-ud-train.sent_split,en_partut-ud-train.csv,4748,1779,2969,234656,120


In [9]:
summary_path = PROCESSED_ROOT / "processing_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")

print("saved summary:", summary_path)

saved summary: data\processed\processing_summary.csv


In [10]:
df_train = pd.read_csv("./data/processed/UD_English-EWT/en_ewt-ud-train.csv")
df_train.head()

,doc_id,char_idx,punct,left_context,right_context,centered_context,token_before,token_after,label
0,en_ewt-ud-train,77,\n,"killed Shaikh Abdullah al-Ani, the preacher at...","mosque in the town of Qaim, near the Syrian bo...","killed Shaikh Abdullah al-Ani, the preacher at...",the,mosque,0
1,en_ewt-ud-train,128,.,"mosque in the town of Qaim, near the Syrian bo...",[This killing of a respected\ncleric will be ...,"mosque in the town of Qaim, near the Syrian bo...",border,NaN,1
2,en_ewt-ud-train,158,\n,ar the Syrian border. [This killing of a respe...,cleric will be causing us trouble for years to...,ar the Syrian border. [This killing of a respe...,respected,cleric,0
3,en_ewt-ud-train,210,.,leric will be causing us trouble for years to ...,] DPA: Iraqi authorities\nannounced that they ...,leric will be causing us trouble for years to ...,come,NaN,1
4,en_ewt-ud-train,235,\n,trouble for years to come.] DPA: Iraqi authori...,announced that they had busted up 3 terrorist ...,trouble for years to come.] DPA: Iraqi authori...,authorities,announced,0


In [11]:
summary_df["positive_ratio"] = summary_df["n_positive"] / summary_df["n_rows"]
summary_df[["dataset", "raw_file", "n_rows", "n_positive", "n_negative", "positive_ratio"]]

,dataset,raw_file,n_rows,n_positive,n_negative,positive_ratio
0,UD_English-EWT,en_ewt-ud-dev.sent_split,5035,1915,3120,0.380338
1,UD_English-EWT,en_ewt-ud-test.sent_split,4994,1963,3031,0.393072
2,UD_English-EWT,en_ewt-ud-train.sent_split,30337,12004,18333,0.395688
3,UD_English-GUM,en_gum-ud-dev.sent_split,4217,1566,2651,0.371354
4,UD_English-GUM,en_gum-ud-test.sent_split,4126,1454,2672,0.352399
5,UD_English-GUM,en_gum-ud-train.sent_split,27578,10204,17374,0.370005
6,UD_English-PUD,en_pud-ud-test.sent_split,3064,1000,2064,0.326371
7,UD_English-ParTUT,en_partut-ud-dev.sent_split,336,156,180,0.464286
8,UD_English-ParTUT,en_partut-ud-test.sent_split,390,153,237,0.392308
9,UD_English-ParTUT,en_partut-ud-train.sent_split,4748,1779,2969,0.374684
